# Notebook 3: Spark Feature Engineering

This notebook demonstrates how selected feature-engineering transformations from the customer churn project are implemented using PySpark DataFrame APIs.

The objective is to translate feature-engineering logic developed in Notebook 1 into Spark while keeping the implementation focused on a small number of meaningful transformations. No model training or complete ML workflow is performed in Spark.

## Notebook Contents

1. Task 1: Load the Customer Dataset using Spark
2. Task 2: Select Feature-Engineering Transformations
3. Task 3: Implement Transformations using PySpark
4. Task 4: Inspect the Resulting Schema
5. Task 5: Display Transformed Data
6. Task 6: Compare Pandas/scikit-learn and PySpark
7. Task 7: Why Spark is Useful at Scale

## Task 1: Load the Customer Dataset using Spark

The customer churn dataset is loaded into a PySpark DataFrame to demonstrate how the feature-engineering workflow can be implemented using Spark.

The raw dataset is loaded without applying any transformations at this stage.

In [1]:
%pip install pyspark

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd

df = pd.read_excel(
    "dataset/Telco_customer_churn.xlsx"
)

df.to_csv(
    "dataset/Telco_customer_churn.csv",
    index=False
)

print("CSV dataset created successfully.")

CSV dataset created successfully.


In [3]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("CustomerChurnFeatureEngineering")
    .getOrCreate()
)

df_spark = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load("dataset/Telco_customer_churn.csv")
)

print("Spark DataFrame loaded successfully.")
print("Number of rows:", df_spark.count())
print("Number of columns:", len(df_spark.columns))

Spark DataFrame loaded successfully.
Number of rows: 7043
Number of columns: 33


In [4]:
df_spark.show(5, truncate=False)

+----------+-----+-------------+----------+-----------+--------+----------------------+---------+-----------+------+--------------+-------+----------+-------------+-------------+--------------+----------------+---------------+-------------+-----------------+------------+------------+----------------+--------------+-----------------+-------------------------+---------------+-------------+-----------+-----------+-----------+----+-----------------------------+
|CustomerID|Count|Country      |State     |City       |Zip Code|Lat Long              |Latitude |Longitude  |Gender|Senior Citizen|Partner|Dependents|Tenure Months|Phone Service|Multiple Lines|Internet Service|Online Security|Online Backup|Device Protection|Tech Support|Streaming TV|Streaming Movies|Contract      |Paperless Billing|Payment Method           |Monthly Charges|Total Charges|Churn Label|Churn Value|Churn Score|CLTV|Churn Reason                 |
+----------+-----+-------------+----------+-----------+--------+------------

## Task 2: Select Feature-Engineering Transformations

Two feature-engineering transformations from Notebook 1 are selected for PySpark implementation.

1. **Average Monthly Spend** — a ratio feature derived from Total Charges and Tenure Months.
2. **Tenure Category** — a binning feature that groups customers according to their tenure duration.

These transformations were selected because they represent meaningful customer-level features and demonstrate how derived features can be created using PySpark DataFrame operations.

## Task 3: Implement Transformations using PySpark


## 3.1 Average Monthly Spend

Average Monthly Spend is implemented as a ratio feature using Total Charges and Tenure Months.

It provides an approximate measure of the customer's average monthly spending over the observed tenure.

In [5]:
from pyspark.sql import functions as F

df_spark = df_spark.withColumn(
    "Total Charges",
    F.col("Total Charges").cast("double")
)

df_spark = df_spark.withColumn(
    "Avg_Monthly_Spend",
    F.when(
        F.col("Tenure Months") > 0,
        F.col("Total Charges") / F.col("Tenure Months")
    ).otherwise(None)
)

df_spark.select(
    "CustomerID",
    "Total Charges",
    "Tenure Months",
    "Avg_Monthly_Spend"
).show(5, truncate=False)

+----------+-------------+-------------+------------------+
|CustomerID|Total Charges|Tenure Months|Avg_Monthly_Spend |
+----------+-------------+-------------+------------------+
|3668-QPYBK|108.15       |2            |54.075            |
|9237-HQITU|151.65       |2            |75.825            |
|9305-CDSKC|820.5        |8            |102.5625          |
|7892-POOKP|3046.05      |28           |108.78750000000001|
|0280-XJGEX|5036.3       |49           |102.78163265306122|
+----------+-------------+-------------+------------------+
only showing top 5 rows


## Task 3.2 Tenure Category

Tenure Category is a binning feature that groups customers into different lifecycle stages based on their tenure duration.

The same customer-tenure grouping used in Notebook 1 is reimplemented here using PySpark DataFrame operations.

In [6]:
df_spark = df_spark.withColumn(
    "Tenure_Category",
    F.when(
        F.col("Tenure Months") <= 12,
        "New"
    ).when(
        F.col("Tenure Months") <= 36,
        "Medium"
    ).otherwise(
        "Long"
    )
)

df_spark.select(
    "CustomerID",
    "Tenure Months",
    "Tenure_Category"
).show(
    5,
    truncate=False
)

+----------+-------------+---------------+
|CustomerID|Tenure Months|Tenure_Category|
+----------+-------------+---------------+
|3668-QPYBK|2            |New            |
|9237-HQITU|2            |New            |
|9305-CDSKC|8            |New            |
|7892-POOKP|28           |Medium         |
|0280-XJGEX|49           |Long           |
+----------+-------------+---------------+
only showing top 5 rows


## Task 4: Inspect the Resulting Schema

The resulting Spark DataFrame schema is inspected to verify the data types of the original columns and the newly created feature-engineered columns.

This confirms that the selected transformations have been successfully incorporated into the PySpark DataFrame.

In [7]:
df_spark.printSchema()

root
 |-- CustomerID: string (nullable = true)
 |-- Count: integer (nullable = true)
 |-- Country: string (nullable = true)
 |-- State: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Zip Code: integer (nullable = true)
 |-- Lat Long: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Senior Citizen: string (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- Tenure Months: integer (nullable = true)
 |-- Phone Service: string (nullable = true)
 |-- Multiple Lines: string (nullable = true)
 |-- Internet Service: string (nullable = true)
 |-- Online Security: string (nullable = true)
 |-- Online Backup: string (nullable = true)
 |-- Device Protection: string (nullable = true)
 |-- Tech Support: string (nullable = true)
 |-- Streaming TV: string (nullable = true)
 |-- Streaming Movies: string (nullable = true)
 |-- Contract:

## Task 5: Display a Sample of the Transformed Data

A sample of the transformed Spark DataFrame is displayed to verify the values of the newly created features alongside the original customer attributes.

In [8]:
df_spark.select(
    "CustomerID",
    "Tenure Months",
    "Total Charges",
    "Avg_Monthly_Spend",
    "Tenure_Category"
).show(10, truncate=False)

+----------+-------------+-------------+------------------+---------------+
|CustomerID|Tenure Months|Total Charges|Avg_Monthly_Spend |Tenure_Category|
+----------+-------------+-------------+------------------+---------------+
|3668-QPYBK|2            |108.15       |54.075            |New            |
|9237-HQITU|2            |151.65       |75.825            |New            |
|9305-CDSKC|8            |820.5        |102.5625          |New            |
|7892-POOKP|28           |3046.05      |108.78750000000001|Medium         |
|0280-XJGEX|49           |5036.3       |102.78163265306122|Long           |
|4190-MFLUW|10           |528.35       |52.835            |New            |
|8779-QRDMV|1            |39.65        |39.65             |New            |
|1066-JKSGK|1            |20.15        |20.15             |New            |
|6467-CHFZW|47           |4749.15      |101.04574468085106|Long           |
|8665-UTDHZ|1            |30.2         |30.2              |New            |
+----------+

## Task 6: Compare Pandas/scikit-learn and PySpark

The same feature-engineering logic was implemented using Pandas/scikit-learn in Notebook 1 and PySpark DataFrame APIs in this notebook.

Both approaches produce the same derived features, but their execution models differ. Pandas/scikit-learn is typically suited to data that fits on a single machine, while PySpark is designed for distributed processing across multiple machines.

## 6.1 Implementation Comparison

| Aspect | Pandas/scikit-learn | PySpark |
|---|---|---|
| Data structure | Pandas DataFrame | Spark DataFrame |
| Average Monthly Spend | Column arithmetic | `withColumn()` and Spark expressions |
| Tenure Category | `pd.cut()` | `when()` / `otherwise()` |
| Execution | Single-machine | Distributed |
| Typical use | Small to medium datasets | Large datasets |

## Task 7: Why Spark Is Useful at Scale

Spark becomes useful when the dataset is too large to be processed efficiently on a single machine.

PySpark can distribute data processing across multiple machines, allowing large datasets and feature-engineering operations to be handled in parallel. This makes Spark suitable for scalable data-processing workflows that may exceed the memory or processing capacity of a single system.